# E5.1 — Portfolio Characterization Along the Pareto Front

Analyses how portfolio composition, development timing, and well count decisions change across the trade-off between expected value and downside risk.

This notebook produces the figure suite for the illustrative DSBPSO run from E4.1: objective-space scatter (with archive-context zoom), field-host network per representative, summary table, production + capacity utilization per representative, ROV distribution per representative, and exercise-timing heatmap per representative.

In [ ]:
import sys
sys.path.insert(0, "..")

from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist

from src.experiments.runner import load_experiment_results, load_run_result
from src.experiments.metrics import hypervolume_2d
from src.optimizer.encoding import decode_particle
from src.optimizer.evaluation import evaluate_portfolio
from src.optimizer.utils import get_non_dominated
from src.data_generation.generator import generate_data
from src.data_generation.profiles import logistic_watercut, make_oil_profile
from src.price.schwartz_smith import generate_price_scenarios_final
from src.valuation.capacity import compute_capped_production
from src.valuation.npv import compute_field_npvs
from src.valuation.lsmc import lsmc_field

EXPERIMENT_ID = "E4.1"
CONFIG_NAME = "dsbpso"

## 1. Load Results and Pick the Illustrative Run

All E4.1 runs solve the same problem instance (`master_seed` is fixed). The 20 seeds only change PSO randomness. We pick the **median-HVI** run as the illustrative case for the portfolio-level figures, and we keep the other 19 runs around for the cross-run cloud and the cross-seed consistency check.

In [ ]:
runs = load_experiment_results(EXPERIMENT_ID)
target_runs = [
    r for r in runs
    if r["summary"]["config_name"] == CONFIG_NAME
    and r["summary"]["status"] == "completed"
]
print(f"Found {len(target_runs)} completed {CONFIG_NAME} runs in {EXPERIMENT_ID}")

all_results = [load_run_result(r["pickle_path"]) for r in target_runs]


def pareto_obj(res):
    pf = res["pareto_front"]
    if not pf:
        return np.empty((0, 2))
    return np.array([e.get("objectives_final", e["objectives"]) for e in pf])


all_fronts = [pareto_obj(r) for r in all_results]

all_pts = np.vstack([f for f in all_fronts if len(f) > 0])
ref_point = np.array([all_pts[:, 0].min() * 0.9, all_pts[:, 1].min() * 0.9])
hvi_values = np.array([
    hypervolume_2d(f, ref_point) if len(f) > 0 else 0.0
    for f in all_fronts
])

median_hvi = float(np.median(hvi_values))
best_idx = int(np.argmin(np.abs(hvi_values - median_hvi)))

result = all_results[best_idx]
config = result["config"]
pareto_front = result["pareto_front"]
objectives = pareto_obj(result)

print(f"Selected illustrative run: seed={target_runs[best_idx]['summary']['seed']}, "
      f"HVI={hvi_values[best_idx]:.1f} (median {median_hvi:.1f}), "
      f"|Pareto|={len(pareto_front)}")

## 2. Decode All Pareto Solutions and Rebuild the Problem Instance

We rebuild `FieldData` / `HostData` / `CostData` from the stored config and generate the high-scenario price set used for re-evaluation. Decoded `PortfolioAllocation`s are kept indexed in the same order as `pareto_front`. We use compact display names (`F1..F9`, `H1..H3`) for figures and tables.

In [ ]:
encoding = config["pso"].get("encoding", None)
n_fields = config["n_fields"]
n_hosts = config["n_hosts"]
n_years = config["n_years"]
max_start_year = config["max_start_year"]
max_wells = config["data_generation"]["max_wells"]

allocations = [
    decode_particle(
        e["position"], n_fields, n_hosts,
        max_start_year, max_wells, encoding=encoding,
    )
    for e in pareto_front
]

field_data, host_data, cost_data = generate_data(config)
prices_final = generate_price_scenarios_final(config)

field_names = [f"F{i + 1}" for i in range(n_fields)]
host_names = [f"H{i + 1}" for i in range(n_hosts)]

print(f"Decoded {len(allocations)} Pareto solutions")
print(f"Problem: {n_fields} fields, {n_hosts} hosts, "
      f"{n_years}-year horizon, max_start_year={max_start_year}, "
      f"max_wells={max_wells}")
print(f"Final price scenarios: {prices_final.n_scenarios}")

## 3. Select Representative Solutions

Three representatives along the front: **Max-ROV** (aggressive), **Balanced** (best compromise — point with the largest perpendicular distance from the line connecting the two extremes), **Max-CVaR** (conservative). All downstream portfolio-level figures key off these.

In [ ]:
def select_representatives(obj):
    n = len(obj)
    if n == 0:
        return {}
    max_rov_idx = int(np.argmax(obj[:, 0]))
    max_cvar_idx = int(np.argmax(obj[:, 1]))
    if n == 1 or max_rov_idx == max_cvar_idx:
        return {"Solution": max_rov_idx}
    reps = {"Max-ROV": max_rov_idx}
    p1, p2 = obj[max_rov_idx], obj[max_cvar_idx]
    line = p2 - p1
    L = np.linalg.norm(line)
    if L > 0 and n >= 3:
        u = line / L
        d = p1 - obj
        cross_vals = np.abs(u[0] * d[:, 1] - u[1] * d[:, 0])
        cross_vals[max_rov_idx] = -1
        cross_vals[max_cvar_idx] = -1
        balanced_idx = int(np.argmax(cross_vals))
        if balanced_idx not in (max_rov_idx, max_cvar_idx):
            reps["Balanced"] = balanced_idx
    reps["Max-CVaR"] = max_cvar_idx
    return reps


representatives = select_representatives(objectives)
slug_map = {
    "Max-ROV": "max_rov",
    "Balanced": "balanced",
    "Max-CVaR": "max_cvar",
    "Solution": "solution",
}

print("Representative solutions:")
for label, idx in representatives.items():
    obj = objectives[idx]
    print(f"  {label:9s} (idx {idx:3d}): "
          f"E[ROV]={obj[0]:10.1f} MNOK   CVaR={obj[1]:10.1f} MNOK")

## 4. Re-evaluate Representatives at `n_scenarios_final`

For each representative we replicate the optimizer's evaluation pipeline on the larger price scenario set to harvest:

- **Per-scenario portfolio ROV** (`scenario_values`) — input for the ROV distribution figure.
- **Per-field exercise year arrays** — input for the exercise-timing figure.
- **Capped per-host oil/gas/water profiles** — input for the production-and-capacity figure.

This is the only computation-heavy cell. Results are cached in `rep_data` for the rest of the notebook.

In [ ]:
val_cfg = config["valuation"]
dg_cfg = config["data_generation"]
per_well_rate = dg_cfg["per_well_rate"]
cutoff_fraction = dg_cfg.get("production_cutoff_fraction", 0.0)
t_arr = np.arange(n_years, dtype=float)


def harvest_representative(alloc):
    res = evaluate_portfolio(
        alloc, field_data, host_data, cost_data, prices_final, config,
        use_lsmc=True, use_capping=True,
    )

    oil_profiles = np.zeros((n_fields, n_years))
    max_rates = np.zeros(n_fields)
    for f in range(n_fields):
        if alloc.assignment[f] < 0:
            continue
        nw = int(alloc.n_wells[f])
        max_rates[f] = nw * per_well_rate * field_data.complexity_factor[f]
        oil_profiles[f] = make_oil_profile(
            n_years=n_years,
            ultimate_recovery=float(field_data.ultimate_recovery[f]),
            max_rate=max_rates[f],
            ramp_years=int(field_data.ramp_years[f]),
            decline_rate=float(field_data.decline_rate[f]),
            cutoff_fraction=cutoff_fraction,
        )

    wc_fractions = np.zeros((n_fields, n_years))
    for f in range(n_fields):
        if alloc.assignment[f] < 0:
            continue
        wc_fractions[f] = logistic_watercut(
            t_arr,
            float(field_data.wc_midpoint[f]),
            float(field_data.wc_initial[f]),
            float(field_data.wc_final[f]),
            float(field_data.wc_steepness[f]),
        )

    capped = compute_capped_production(
        oil_profiles, host_data, alloc, n_years,
        max_rates=max_rates,
        ramp_years=field_data.ramp_years,
        gor=field_data.gor,
        wc_fractions=wc_fractions,
    )

    exercise = {}
    for host_idx, host_block in capped.items():
        n_on = len(host_block["field_indices"])
        shared_opex = float(cost_data.opex_rate[host_idx]) / n_on
        for li, fi in enumerate(host_block["field_indices"]):
            start = int(alloc.start_year[fi])
            nw = int(alloc.n_wells[fi])
            capex = (
                float(cost_data.well_cost_per_well[fi]) * nw
                + cost_data.subsea_base
                + float(cost_data.pipeline_cost[fi, host_idx])
            )
            npvs = compute_field_npvs(
                capped_oil=host_block["oil"][li],
                capped_gas=host_block["gas"][li],
                oil_prices=prices_final.oil,
                gas_prices=prices_final.gas,
                capex=capex,
                opex_rate=shared_opex,
                variable_opex_rate=float(cost_data.variable_opex_rate[host_idx]),
                abex=float(cost_data.abex[host_idx]),
                discount_rate=val_cfg["discount_rate"],
                start_year=start,
                nok_per_usd=val_cfg["nok_per_usd"],
                barrel=val_cfg["barrel"],
                btu=val_cfg["btu"],
            )
            _, ex = lsmc_field(
                npvs=npvs,
                oil_prices=prices_final.oil,
                gas_prices=prices_final.gas,
                discount_rate=val_cfg["discount_rate"],
                start_year=start,
            )
            exercise[fi] = ex

    return {
        "valuation": res,
        "capped": capped,
        "exercise_year": exercise,
        "oil_profiles": oil_profiles,
    }


rep_data = {label: harvest_representative(allocations[idx])
            for label, idx in representatives.items()}

print("Re-evaluation sanity (re-eval mean vs stored objectives_final):")
for label, idx in representatives.items():
    pf_obj = pareto_front[idx].get("objectives_final", pareto_front[idx]["objectives"])
    re_mean = float(rep_data[label]["valuation"].scenario_values.mean())
    diff_pct = (re_mean - pf_obj[0]) / max(1.0, abs(pf_obj[0])) * 100
    print(f"  {label:9s}: stored E[ROV]={pf_obj[0]:10.1f}, "
          f"re-eval mean={re_mean:10.1f}, diff={diff_pct:+.2f}%")

## 5. Figure 1 — Pareto front and candidate portfolios explored

Two figures (axes: x = CVaR$_{0.05}$, y = Expected ROV, both maximized so the upper-right is the ideal).

- Full view: combined **non-dominated candidate portfolios** across all 20 DSBPSO runs of this instance (grey cloud), with the illustrative run's Pareto front overlaid as red stars.
- Zoom on the front, **illustrative run only**: red stars plus the dominated candidates from that single PSO run (silver dots).

In [ ]:
def archive_obj(res):
    arch = res.get("archive", [])
    if not arch:
        return np.empty((0, 2))
    return np.array([e.get("objectives_final", e["objectives"]) for e in arch])


# Combined archive across all runs, then split into non-dominated vs dominated
all_archive = np.vstack([archive_obj(r) for r in all_results])
nd_mask = get_non_dominated(all_archive, maximize=(True, True))
nondom_cloud = all_archive[nd_mask]
all_dominated = all_archive[~nd_mask]
print(f"Across {len(all_results)} runs: "
      f"{len(nondom_cloud)} non-dominated, "
      f"{len(all_dominated)} dominated "
      f"(total archive {len(all_archive)})")

# Illustrative run's archive (split into non-dominated vs dominated for that run only)
illustrative_archive = archive_obj(result)
illustrative_nd = get_non_dominated(illustrative_archive, maximize=(True, True))
illustrative_dominated = illustrative_archive[~illustrative_nd]

# Distinct colors for the three representative portfolios
rep_colors = {
    "Max-ROV":   "dodgerblue",
    "Balanced":  "seagreen",
    "Max-CVaR":  "goldenrod",
}
rep_indices = list(representatives.values())
rep_mask = np.zeros(len(objectives), dtype=bool)
for ri in rep_indices:
    rep_mask[ri] = True
other_obj = objectives[~rep_mask]


# ── Figure 1: full view ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 7))
ax.scatter(nondom_cloud[:, 1], nondom_cloud[:, 0],
           s=14, color="lightgray", alpha=0.6, edgecolors="none",
           label=f"Non-dominated candidate portfolios "
                 f"(across all {len(all_results)} runs, "
                 f"{len(nondom_cloud)} portfolios)")
ax.scatter(other_obj[:, 1], other_obj[:, 0],
           s=35, c="crimson", marker="o", edgecolors="black",
           linewidth=0.4, zorder=5,
           label=f"Pareto front ({len(objectives)} portfolios)")
for label, idx in representatives.items():
    ax.scatter(objectives[idx, 1], objectives[idx, 0],
               s=160, c=rep_colors[label], marker="*",
               edgecolors="black", linewidth=0.4, zorder=6,
               label=label)
ax.set_xlabel(r"CVaR$_{0.05}$ (MNOK)")
ax.set_ylabel("Expected ROV (MNOK)")
ax.set_title("Pareto front and non-dominated candidate portfolios")
ax.legend(loc="lower left")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


# ── Figure 2: zoom — illustrative run only ──────────────────────
fig, ax = plt.subplots(figsize=(9, 7))
if len(illustrative_dominated) > 0:
    ax.scatter(illustrative_dominated[:, 1], illustrative_dominated[:, 0],
               s=32, color="silver", alpha=0.75,
               edgecolors="dimgray", linewidth=0.4,
               label=f"Dominated candidate portfolios "
                     f"({len(illustrative_dominated)} portfolios)")
ax.scatter(other_obj[:, 1], other_obj[:, 0],
           s=40, c="crimson", marker="o", edgecolors="black",
           linewidth=0.5, zorder=5,
           label=f"Pareto front ({len(objectives)} portfolios)")
for label, idx in representatives.items():
    ax.scatter(objectives[idx, 1], objectives[idx, 0],
               s=180, c=rep_colors[label], marker="*",
               edgecolors="black", linewidth=0.5, zorder=6,
               label=label)

combined = np.vstack([illustrative_archive, objectives])
xpad = 0.05 * (combined[:, 1].max() - combined[:, 1].min() + 1)
ypad = 0.05 * (combined[:, 0].max() - combined[:, 0].min() + 1)
ax.set_xlim(combined[:, 1].min() - xpad, combined[:, 1].max() + xpad)
ax.set_ylim(combined[:, 0].min() - ypad, combined[:, 0].max() + ypad)

ax.set_xlabel(r"CVaR$_{0.05}$ (MNOK)")
ax.set_ylabel("Expected ROV (MNOK)")
ax.set_title("Pareto front and dominated candidate portfolios")
ax.legend(loc="lower left")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 6. Figure 2 — Field–host network diagrams (one per representative)

Spatial map per representative, drawn in the same style as the spatial map in notebook 01:

- **All** field-host pairs drawn as faint grey lines (the static instance backbone), unlabelled.
- **Selected tiebacks** overlaid as colored bold lines, one color per host, labelled with the routed distance in km.
- Fields = blue circles, hosts = red squares, names in bold next to each marker.
- Axes auto-zoomed to the actual point cloud so the layout fills the figure.

In [ ]:
dist_matrix = cdist(field_data.coords, host_data.coords)
host_palette = ["steelblue", "tomato", "seagreen", "darkorange", "purple", "goldenrod"]


def plot_tieback_map(ax, alloc, title=None):
    # Selected tiebacks (colored by host) annotated with km distance
    # Place label 35% along the line from the field, plus a perpendicular
    # offset so labels on lines sharing an endpoint don't overlap.
    all_xy = np.vstack([field_data.coords, host_data.coords])
    map_extent = max(
        all_xy[:, 0].max() - all_xy[:, 0].min(),
        all_xy[:, 1].max() - all_xy[:, 1].min(),
    )
    perp_offset = 0.025 * map_extent

    for f in range(n_fields):
        h = int(alloc.assignment[f])
        if h < 0:
            continue
        fx, fy = field_data.coords[f]
        hx, hy = host_data.coords[h]
        color = host_palette[h % len(host_palette)]
        ax.plot([fx, hx], [fy, hy], "-", color=color, lw=2.4, alpha=0.9, zorder=3)

        t = 0.35
        mx, my = fx + t * (hx - fx), fy + t * (hy - fy)
        dx, dy = hx - fx, hy - fy
        L = float(np.hypot(dx, dy))
        if L > 0:
            mx += perp_offset * (-dy / L)
            my += perp_offset * (dx / L)
        ax.annotate(
            f"{dist_matrix[f, h]:.0f} km",
            (mx, my), fontsize=8, ha="center", va="center", zorder=4,
            bbox=dict(boxstyle="round,pad=0.25", fc="white", ec=color, alpha=0.95),
        )

    # Fields
    ax.scatter(field_data.coords[:, 0], field_data.coords[:, 1],
               s=120, c="steelblue", marker="o", zorder=5,
               edgecolors="white", linewidth=1.5)
    for f, name in enumerate(field_names):
        h = int(alloc.assignment[f])
        label = name if h >= 0 else f"{name} (skip)"
        ax.annotate(label, field_data.coords[f], fontsize=9, fontweight="bold",
                    textcoords="offset points", xytext=(8, 8), color="steelblue")

    # Hosts
    ax.scatter(host_data.coords[:, 0], host_data.coords[:, 1],
               s=200, c="tomato", marker="s", zorder=5,
               edgecolors="white", linewidth=1.5)
    for h, name in enumerate(host_names):
        ax.annotate(name, host_data.coords[h], fontsize=9, fontweight="bold",
                    textcoords="offset points", xytext=(8, 8), color="tomato")

    # Square axes: equal x/y extent centred on the point cloud
    x_min, x_max = all_xy[:, 0].min(), all_xy[:, 0].max()
    y_min, y_max = all_xy[:, 1].min(), all_xy[:, 1].max()
    half = max(x_max - x_min, y_max - y_min) / 2 + 0.08 * map_extent
    xc, yc = (x_min + x_max) / 2, (y_min + y_max) / 2
    ax.set_xlim(xc - half, xc + half)
    ax.set_ylim(yc - half, yc + half)

    ax.set_xlabel("x (km)")
    ax.set_ylabel("y (km)")
    if title is not None:
        ax.set_title(title)
    ax.set_aspect("equal")


for label, idx in representatives.items():
    fig, ax = plt.subplots(figsize=(8, 8))
    plot_tieback_map(ax, allocations[idx])
    plt.tight_layout()
    plt.show()

## 6b. Topology comparison — Balanced vs Max-CVaR

Max-CVaR uses a different host assignment from the rest of the Pareto front: F3 is rerouted from Host 3 to Host 2. This cell generates side-by-side network maps and an objective-space highlight to visualise the topology change and its effect on the objectives.

In [ ]:
# --- Topology summary ---
def assignment_signature(alloc):
    """Hashable representation of just the host assignment (ignores timing/wells)."""
    return tuple(int(x) for x in alloc.assignment)


pareto_signatures = [assignment_signature(a) for a in allocations]
unique_sigs = set(pareto_signatures)
print(f"Pareto front spans {len(unique_sigs)} distinct topology/topologies.")

balanced_label = "Balanced" if "Balanced" in representatives else "Max-ROV"
balanced_idx = representatives[balanced_label]
max_cvar_idx = representatives["Max-CVaR"]

ref_alloc = allocations[balanced_idx]
cvar_alloc = allocations[max_cvar_idx]

balanced_obj = objectives[balanced_idx]
cvar_obj = objectives[max_cvar_idx]

print(f"Balanced:  {assignment_signature(ref_alloc)}")
print(f"Max-CVaR:  {assignment_signature(cvar_alloc)}")

# Identify field-host differences
diffs = []
for f in range(n_fields):
    ref_h = int(ref_alloc.assignment[f])
    cvar_h = int(cvar_alloc.assignment[f])
    if ref_h != cvar_h:
        ref_label = host_names[ref_h] if ref_h >= 0 else "skip"
        cvar_label = host_names[cvar_h] if cvar_h >= 0 else "skip"
        diffs.append((field_names[f], ref_label, cvar_label))

print(f"\nField-host changes Balanced → Max-CVaR ({len(diffs)} field(s)):")
for fname, ref_label, cvar_label in diffs:
    print(f"  {fname}: {ref_label} → {cvar_label}")


def total_pipeline_km(alloc):
    total = 0.0
    for f in range(n_fields):
        h = int(alloc.assignment[f])
        if h < 0:
            continue
        total += float(dist_matrix[f, h])
    return total


ref_km = total_pipeline_km(ref_alloc)
cvar_km = total_pipeline_km(cvar_alloc)
print(f"\nTotal pipeline length:")
print(f"  Balanced:  {ref_km:6.1f} km")
print(f"  Max-CVaR:  {cvar_km:6.1f} km   (delta = {cvar_km - ref_km:+.1f} km)")

print(f"\nPer-host fields (Balanced):")
for h in range(n_hosts):
    fl = [field_names[f] for f in range(n_fields) if ref_alloc.assignment[f] == h]
    print(f"  {host_names[h]}: {fl}")
print(f"\nPer-host fields (Max-CVaR):")
for h in range(n_hosts):
    fl = [field_names[f] for f in range(n_fields) if cvar_alloc.assignment[f] == h]
    print(f"  {host_names[h]}: {fl}")

In [ ]:
# Side-by-side network maps: Balanced vs Max-CVaR
fig, axes = plt.subplots(1, 2, figsize=(15, 7.5))

plot_tieback_map(
    axes[0], ref_alloc,
    f"Balanced — E[ROV]={balanced_obj[0]:.0f}, CVaR={balanced_obj[1]:.0f} MNOK\n"
    f"({ref_km:.0f} km total pipeline)",
)
plot_tieback_map(
    axes[1], cvar_alloc,
    f"Max-CVaR — E[ROV]={cvar_obj[0]:.0f}, CVaR={cvar_obj[1]:.0f} MNOK\n"
    f"({cvar_km:.0f} km total pipeline; {len(diffs)} field rerouted)",
)
plt.tight_layout()
plt.show()


# Highlight representatives on the zoomed objective-space scatter (illustrative run)
fig, ax = plt.subplots(figsize=(9, 7))
if len(illustrative_dominated) > 0:
    ax.scatter(illustrative_dominated[:, 1], illustrative_dominated[:, 0],
               s=24, color="silver", alpha=0.55, edgecolors="none",
               label="Dominated candidates")
ax.scatter(other_obj[:, 1], other_obj[:, 0],
           s=40, c="crimson", marker="o", edgecolors="black",
           linewidth=0.5, zorder=5,
           label=f"Pareto front ({len(objectives)} portfolios)")
for label, idx in representatives.items():
    ax.scatter(objectives[idx, 1], objectives[idx, 0],
               s=180, c=rep_colors[label], marker="*",
               edgecolors="black", linewidth=0.5, zorder=6,
               label=label)

focus = np.vstack([illustrative_archive, objectives])
xpad = 0.05 * (focus[:, 1].max() - focus[:, 1].min() + 1)
ypad = 0.05 * (focus[:, 0].max() - focus[:, 0].min() + 1)
ax.set_xlim(focus[:, 1].min() - xpad, focus[:, 1].max() + xpad)
ax.set_ylim(focus[:, 0].min() - ypad, focus[:, 0].max() + ypad)

ax.set_xlabel(r"CVaR$_{0.05}$ (MNOK)")
ax.set_ylabel("Expected ROV (MNOK)")
ax.set_title("Balanced vs Max-CVaR on the Pareto front")
ax.legend(loc="lower left")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 7. Pareto portfolio summary table

Compact summary printed as a LaTeX table with multirow grouping. Columns: Solution, Host, Fields, $E[\mathrm{ROV}]$ (MNOK), $\mathrm{CVaR}_{0.05}$ (MNOK).

In [ ]:
def build_summary_latex(representatives, allocations, objectives):
    sol_blocks = []
    for label, idx in representatives.items():
        alloc = allocations[idx]
        obj = objectives[idx]
        host_rows = []
        for h in range(n_hosts):
            fields_on = [f for f in range(n_fields) if alloc.assignment[f] == h]
            if not fields_on:
                continue
            host_rows.append({
                "host": host_names[h],
                "fields": ", ".join(field_names[f] for f in fields_on),
            })
        sol_blocks.append({
            "label": label,
            "rows": host_rows,
            "rov": f"{obj[0]:.0f}",
            "cvar": f"{obj[1]:.0f}",
        })

    lines = [
        r"\begin{table}[htbp]",
        r"    \centering",
        r"    \caption{Pareto-optimal portfolios.}",
        r"    \label{tab:e5_1_portfolio_summary}",
        r"    \begin{tabular}{llllc}",
        r"        \toprule",
        r"        Solution & Host & Fields & $E[\mathrm{ROV}]$ (MNOK) & $\mathrm{CVaR}_{0.05}$ (MNOK) \\",
        r"        \midrule",
    ]
    for bi, block in enumerate(sol_blocks):
        n_rows = len(block["rows"])
        for ri, row in enumerate(block["rows"]):
            if ri == 0:
                lines.append(
                    f"        \\multirow{{{n_rows}}}{{*}}{{{block['label']}}} "
                    f"& {row['host']} & {row['fields']} "
                    f"& \\multirow{{{n_rows}}}{{*}}{{{block['rov']}}} "
                    f"& \\multirow{{{n_rows}}}{{*}}{{{block['cvar']}}} \\\\"
                )
            else:
                lines.append(
                    f"        & {row['host']} & {row['fields']} & & \\\\"
                )
        if bi < len(sol_blocks) - 1:
            lines.append(r"        \midrule")
    lines += [
        r"        \bottomrule",
        r"    \end{tabular}",
        r"\end{table}",
    ]
    return "\n".join(lines)


print(build_summary_latex(representatives, allocations, objectives))

## 8. Figure 3 — Production and capacity utilization (one per representative)

Per representative: oil / gas / water rows, one column per active host. Each cell shows per-field stacked capped production from `compute_capped_production` with the host capacity overlaid as a black dashed line.

In [ ]:
# Darker-pastel palette: same hues as the tornado figure but more saturated for
# better contrast when stacked.
PASTEL_FIELD_COLORS = [
    "cornflowerblue", "salmon", "mediumseagreen", "darkkhaki", "tan",
    "darkgray", "indianred", "cadetblue", "plum",
]
field_palette = [PASTEL_FIELD_COLORS[i % len(PASTEL_FIELD_COLORS)]
                 for i in range(max(n_fields, 1))]

years = np.arange(n_years)
streams = [
    ("oil", host_data.oil_capacity, "Oil (MSm$^3$/y)"),
    ("gas", host_data.gas_capacity, "Gas (BSm$^3$/y)"),
    ("water", host_data.water_capacity, "Water (MSm$^3$/y)"),
]


def plot_production_capacity(rep_label, rep_idx, slug):
    capped = rep_data[rep_label]["capped"]
    active_hosts = sorted(capped.keys())
    n_active = len(active_hosts)
    if n_active == 0:
        print(f"{rep_label}: no developed fields, skipping production figure")
        return

    # Rows = hosts, columns = commodities (oil, gas, water).
    # Each field has the same color across every panel.
    fig, axes = plt.subplots(n_active, 3,
                             figsize=(15, 3.5 * n_active),
                             squeeze=False, sharex=True)

    legend_handles = {}

    for row, host_idx in enumerate(active_hosts):
        host_block = capped[host_idx]
        sorted_li = sorted(range(len(host_block["field_indices"])),
                           key=lambda li: host_block["field_indices"][li])
        for col, (key, cap_arr, ylabel) in enumerate(streams):
            ax = axes[row, col]
            stacked = np.zeros(n_years)
            for li in sorted_li:
                fi = host_block["field_indices"][li]
                series = host_block[key][li]
                patch = ax.fill_between(
                    years, stacked, stacked + series,
                    color=field_palette[fi], alpha=0.9,
                    edgecolor="white", linewidth=0.6,
                    label=field_names[fi],
                )
                legend_handles.setdefault(fi, patch)
                stacked += series
            cap_line, = ax.plot(years, cap_arr[host_idx], "k--", lw=2.4, label="Host capacity", zorder=10)
            legend_handles.setdefault("cap", cap_line)
            ax.tick_params(axis="both", labelsize=10)
            if row == 0:
                ax.set_title(ylabel, fontsize=14, fontweight="bold")
            if row == n_active - 1:
                ax.set_xlabel("Year", fontsize=11)
            if col == 0:
                ax.set_ylabel(host_names[host_idx], fontsize=16, fontweight="bold",
                              rotation=0, labelpad=25, va="center")

    field_keys = sorted(k for k in legend_handles if isinstance(k, int))
    ordered = [legend_handles[fi] for fi in field_keys]
    ordered_labels = [field_names[fi] for fi in field_keys]
    if "cap" in legend_handles:
        ordered.append(legend_handles["cap"])
        ordered_labels.append("Host capacity")
    fig.legend(ordered, ordered_labels,
               loc="lower center", ncol=min(len(ordered), 5),
               bbox_to_anchor=(0.5, -0.02), frameon=False, fontsize=13)

    plt.tight_layout(rect=[0, 0.04, 1, 1])
    plt.show()


# Plot only the Balanced portfolio (or the single solution if the front has one point)
if "Balanced" in representatives:
    plot_production_capacity("Balanced", representatives["Balanced"], "balanced")
elif "Solution" in representatives:
    plot_production_capacity("Solution", representatives["Solution"], "solution")

## 9. Figure 4 — ROV distributions (one per representative)

One row per representative, each with two panels: full range and zoom (clipped at the 99th percentile). Vertical reference lines mark **Mean**, **VaR$_{0.05}$** (the threshold below which 5 % of paths fall) and **CVaR$_{0.05}$** (the mean of those worst 5 %). The y-axis is the count of MC paths in each bin.

In [ ]:
import math

rep_color = {
    "Max-ROV": "indianred",
    "Balanced": "cornflowerblue",
    "Max-CVaR": "mediumseagreen",
    "Solution": "cornflowerblue",
}

cvar_alpha = float(val_cfg.get("cvar_alpha", 0.05))


def _round_up_nice(value, target_step=10000):
    if value <= 0:
        return target_step
    return int(math.ceil(value / target_step) * target_step)


def _draw_reference_lines(ax, mean_val, var_val, cvar_val):
    ax.axvline(mean_val, color="black", lw=1.6, linestyle="--",
               label=f"Mean = {mean_val:,.0f}")
    ax.axvline(var_val, color="darkorange", lw=1.6, linestyle="-.",
               label=f"VaR$_{{{cvar_alpha:g}}}$ = {var_val:,.0f}")
    ax.axvline(cvar_val, color="firebrick", lw=1.6, linestyle=":",
               label=f"CVaR$_{{{cvar_alpha:g}}}$ = {cvar_val:,.0f}")


# Order portfolios conservative → aggressive
display_order = [r for r in ["Max-CVaR", "Balanced", "Max-ROV"] if r in representatives]
n_rows = len(display_order)

# Compute a shared zoom x-range across all portfolios so the right column is comparable
zoom_xmax = _round_up_nice(
    max(float(np.percentile(rep_data[r]["valuation"].scenario_values, 99))
        for r in display_order),
    target_step=10000,
)

fig, axes = plt.subplots(n_rows, 2, figsize=(14, 4.2 * n_rows), squeeze=False)

for row, label in enumerate(display_order):
    rep_idx = representatives[label]
    sv = rep_data[label]["valuation"].scenario_values
    color = rep_color.get(label, "cornflowerblue")
    obj = objectives[rep_idx]

    mean_val = float(sv.mean())
    var_val = float(np.percentile(sv, 100 * cvar_alpha))
    tail = sv[sv <= var_val]
    cvar_val = float(tail.mean()) if tail.size > 0 else var_val

    # Left: full range
    ax = axes[row, 0]
    ax.hist(sv, bins=80, color=color, alpha=0.85, edgecolor="white", linewidth=0.3)
    _draw_reference_lines(ax, mean_val, var_val, cvar_val)
    ax.set_ylabel(f"{label}\nNumber of paths", fontsize=11)
    if row == 0:
        ax.set_title("Full range", fontsize=12, fontweight="bold")
    if row == n_rows - 1:
        ax.set_xlabel("Portfolio ROV (MNOK)")
    ax.legend(loc="upper right", fontsize=9)

    # Right: zoom 0..zoom_xmax (shared across rows)
    ax = axes[row, 1]
    sv_in = sv[sv <= zoom_xmax]
    ax.hist(sv_in, bins=60, color=color, alpha=0.85, edgecolor="white", linewidth=0.3)
    _draw_reference_lines(ax, mean_val, var_val, cvar_val)
    ax.set_xlim(left=0, right=zoom_xmax)
    if row == 0:
        ax.set_title(f"Zoomed view (0 to {zoom_xmax:,} MNOK)", fontsize=12, fontweight="bold")
    if row == n_rows - 1:
        ax.set_xlabel("Portfolio ROV (MNOK)")
    ax.legend(loc="upper right", fontsize=9)

plt.tight_layout()
plt.show()


print("ROV distribution summary (Mean / VaR_0.05 / CVaR_0.05):")
for label in display_order:
    sv = rep_data[label]["valuation"].scenario_values
    var_val = float(np.percentile(sv, 100 * cvar_alpha))
    tail = sv[sv <= var_val]
    cvar_val = float(tail.mean()) if tail.size > 0 else var_val
    print(f"  {label:9s}: mean={sv.mean():9.1f}, "
          f"VaR={var_val:9.1f}, CVaR={cvar_val:9.1f}, "
          f"max={sv.max():9.1f}")

## 10. Figure 5 — Exercise outcome summary (one per representative)

Simple horizontal stacked bar per representative. Each row is one developed field-host tieback. The bar splits 100 % of the MC paths into three buckets:

- **Exercised at start year** — the option is taken immediately when first legal.
- **Exercised later** — the option is taken some year after `start_year` (deferral occurred).
- **Never exercised** — the option expires unused over the planning horizon.

This compresses the per-field exercise distribution into the three numbers that matter for portfolio interpretation: how often is the option used immediately, how often deferred, how often abandoned.

In [ ]:
def plot_exercise_summary(rep_label, rep_idx, slug):
    alloc = allocations[rep_idx]
    exercises = rep_data[rep_label]["exercise_year"]
    developed = sorted(
        exercises.keys(), key=lambda f: (int(alloc.start_year[f]), f),
    )
    if not developed:
        print(f"{rep_label}: no developed fields, skipping exercise figure")
        return

    n_dev = len(developed)
    pct_exercised = np.zeros(n_dev)
    pct_never = np.zeros(n_dev)
    row_labels = []

    for i, fi in enumerate(developed):
        ex = exercises[fi]
        start = int(alloc.start_year[fi])
        host = int(alloc.assignment[fi])
        pct_exercised[i] = (ex >= 0).mean() * 100
        pct_never[i] = (ex == -1).mean() * 100
        row_labels.append(
            f"{field_names[fi]} → {host_names[host]} (starts year {start})"
        )

    fig, ax = plt.subplots(figsize=(11, 0.55 * n_dev + 2.2))
    y_pos = np.arange(n_dev)

    ax.barh(y_pos, pct_exercised, color="steelblue", height=0.7,
            label="Exercised", edgecolor="white", linewidth=0.5)
    ax.barh(y_pos, pct_never, left=pct_exercised, color="lightcoral", height=0.7,
            label="Never exercised", edgecolor="white", linewidth=0.5)

    for i in range(n_dev):
        cumulative = 0.0
        for value, color_text in [
            (pct_exercised[i], "white"),
            (pct_never[i], "black"),
        ]:
            if value >= 6:
                ax.text(cumulative + value / 2, i,
                        f"{value:.0f}%", ha="center", va="center",
                        fontsize=9, color=color_text, fontweight="bold")
            cumulative += value

    ax.set_yticks(y_pos)
    ax.set_yticklabels(row_labels)
    ax.invert_yaxis()
    ax.set_xlim(0, 100)
    ax.set_xlabel("Fraction of MC paths (%)")
    ax.legend(loc="lower right", framealpha=0.95)
    ax.grid(True, axis="x", alpha=0.3)

    obj = objectives[rep_idx]
    ax.set_title(
        f"Exercise outcome — {rep_label} "
        f"(E[ROV]={obj[0]:.0f}, CVaR={obj[1]:.0f} MNOK)",
        fontweight="bold", fontsize=11,
    )
    plt.tight_layout()
    plt.show()


for label, idx in representatives.items():
    plot_exercise_summary(label, idx, slug_map.get(
        label, label.lower().replace(" ", "_"),
    ))


## 10b. Figure 5b — Exercise timing buckets (one per representative)

Companion to Figure 5. Same horizontal stacked-bar layout, but the "Exercised" segment is split by how long LSMC deferred relative to the planned `start_year`:

- **At start year** — exercised in the planned year (no deferral).
- **Deferred 1–2y / 3–5y / 6+y** — exercised later than planned by the indicated number of years.
- **Never exercised** — option expires unused.

This shows whether the deferral option is actually used (and by how much) per field, rather than just whether the option was eventually exercised at all.

In [ ]:
def plot_exercise_timing(rep_label, rep_idx, slug):
    alloc = allocations[rep_idx]
    exercises = rep_data[rep_label]["exercise_year"]
    developed = sorted(
        exercises.keys(), key=lambda f: (int(alloc.start_year[f]), f),
    )
    if not developed:
        print(f"{rep_label}: no developed fields, skipping timing figure")
        return

    bucket_labels = [
        "At start year",
        "Deferred 1–2y",
        "Deferred 3–5y",
        "Deferred 6+y",
        "Never exercised",
    ]
    bucket_colors = [
        "#1f4e79",
        "#4a8cc7",
        "#8dc3e6",
        "#d9b58a",
        "lightcoral",
    ]
    text_colors = ["white", "white", "black", "black", "black"]

    n_dev = len(developed)
    pct = np.zeros((n_dev, len(bucket_labels)))
    row_labels = []

    for i, fi in enumerate(developed):
        ex = exercises[fi]
        start = int(alloc.start_year[fi])
        host = int(alloc.assignment[fi])
        delay = np.where(ex >= 0, ex - start, -1)
        n = len(ex)
        pct[i, 0] = (delay == 0).sum() / n * 100
        pct[i, 1] = ((delay >= 1) & (delay <= 2)).sum() / n * 100
        pct[i, 2] = ((delay >= 3) & (delay <= 5)).sum() / n * 100
        pct[i, 3] = (delay >= 6).sum() / n * 100
        pct[i, 4] = (ex == -1).sum() / n * 100
        row_labels.append(
            f"{field_names[fi]} → {host_names[host]} (starts year {start})"
        )

    fig, ax = plt.subplots(figsize=(12, 0.55 * n_dev + 2.6))
    y_pos = np.arange(n_dev)

    left = np.zeros(n_dev)
    for b, (lbl, c) in enumerate(zip(bucket_labels, bucket_colors)):
        ax.barh(y_pos, pct[:, b], left=left, color=c, height=0.7,
                label=lbl, edgecolor="white", linewidth=0.5)
        for i in range(n_dev):
            v = pct[i, b]
            if v >= 6:
                ax.text(left[i] + v / 2, i,
                        f"{v:.0f}%", ha="center", va="center",
                        fontsize=9, color=text_colors[b], fontweight="bold")
        left += pct[:, b]

    ax.set_yticks(y_pos)
    ax.set_yticklabels(row_labels)
    ax.invert_yaxis()
    ax.set_xlim(0, 100)
    ax.set_xlabel("Fraction of MC paths (%)")
    ax.legend(loc="lower center", bbox_to_anchor=(0.5, -0.22),
              ncol=len(bucket_labels), framealpha=0.95, fontsize=9)
    ax.grid(True, axis="x", alpha=0.3)

    obj = objectives[rep_idx]
    ax.set_title(
        f"Exercise timing — {rep_label} "
        f"(E[ROV]={obj[0]:.0f}, CVaR={obj[1]:.0f} MNOK)",
        fontweight="bold", fontsize=11,
    )
    plt.tight_layout()
    plt.show()


for label, idx in representatives.items():
    plot_exercise_timing(label, idx, slug_map.get(
        label, label.lower().replace(" ", "_"),
    ))

## 11. Field core / swing classification

Across the full Pareto front: which fields appear in almost every solution (core), which appear conditionally (swing), and which rarely appear (rare).

In [ ]:
n_solutions = len(allocations)
developed_mat = np.zeros((n_solutions, n_fields))
for i, alloc in enumerate(allocations):
    developed_mat[i] = (alloc.assignment >= 0).astype(float)

freq = developed_mat.mean(axis=0)
core_threshold = 0.8
swing_threshold = 0.2

bar_colors = []
for v in freq:
    if v >= core_threshold:
        bar_colors.append("steelblue")
    elif v >= swing_threshold:
        bar_colors.append("goldenrod")
    else:
        bar_colors.append("lightcoral")

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(range(n_fields), freq, color=bar_colors, edgecolor="black", linewidth=0.5)
ax.axhline(core_threshold, color="steelblue", linestyle="--", alpha=0.5,
           label=f"Core (>{int(core_threshold*100)}%)")
ax.axhline(swing_threshold, color="lightcoral", linestyle="--", alpha=0.5,
           label=f"Rare (<{int(swing_threshold*100)}%)")
ax.set_xticks(range(n_fields))
ax.set_xticklabels(field_names, rotation=0)
ax.set_ylabel("Selection frequency")
ax.set_title("Field selection frequency across Pareto front")
ax.legend()
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

print("\nField classification:")
for i, v in enumerate(freq):
    cat = "CORE" if v >= core_threshold else ("SWING" if v >= swing_threshold else "RARE")
    print(f"  {field_names[i]}: {v*100:5.1f}% — {cat}")

## 12. Timing and well-count heatmaps

Per Pareto solution (sorted by E[ROV] ↓): start year and well count per field. Reveals whether high-ROV portfolios push for earlier development and more wells.

In [ ]:
rov_sort = np.argsort(-objectives[:, 0])

start_years_mat = np.full((n_solutions, n_fields), np.nan)
wells_mat = np.full((n_solutions, n_fields), np.nan)
for i, alloc in enumerate(allocations):
    for f in range(n_fields):
        if alloc.assignment[f] >= 0:
            start_years_mat[i, f] = alloc.start_year[f]
            wells_mat[i, f] = alloc.n_wells[f]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

im1 = axes[0].imshow(start_years_mat[rov_sort], aspect="auto",
                     cmap="viridis", interpolation="nearest")
axes[0].set_xlabel("Field")
axes[0].set_ylabel("Solution (sorted by E[ROV] ↓)")
axes[0].set_xticks(range(n_fields))
axes[0].set_xticklabels(field_names)
axes[0].set_title("Start year (NaN = not developed)")
fig.colorbar(im1, ax=axes[0], label="Start year")

im2 = axes[1].imshow(wells_mat[rov_sort], aspect="auto",
                     cmap="YlOrRd", interpolation="nearest")
axes[1].set_xlabel("Field")
axes[1].set_ylabel("Solution (sorted by E[ROV] ↓)")
axes[1].set_xticks(range(n_fields))
axes[1].set_xticklabels(field_names)
axes[1].set_title("Well count (NaN = not developed)")
fig.colorbar(im2, ax=axes[1], label="Wells")

plt.tight_layout()
plt.show()

## 13. Host utilization (stacked bar across the front)

Number of fields routed to each host per Pareto solution. Aggregate companion to the per-portfolio capacity figure (§8).

In [ ]:
fields_per_host = np.zeros((n_solutions, n_hosts))
for i, alloc in enumerate(allocations):
    for f in range(n_fields):
        h = alloc.assignment[f]
        if h >= 0:
            fields_per_host[i, h] += 1

fig, ax = plt.subplots(figsize=(10, 5))
host_palette_local = ["steelblue", "coral", "seagreen", "orchid", "goldenrod"]
sorted_data = fields_per_host[rov_sort]
for h in range(n_hosts):
    bottom = sorted_data[:, :h].sum(axis=1) if h > 0 else None
    ax.bar(range(n_solutions), sorted_data[:, h],
           bottom=bottom,
           label=host_names[h],
           color=host_palette_local[h % len(host_palette_local)],
           alpha=0.8)

ax.set_xlabel("Solution (sorted by E[ROV] ↓)")
ax.set_ylabel("Fields assigned")
ax.set_title("Host utilization across Pareto front")
ax.legend()
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

print("Average fields per host:")
for h in range(n_hosts):
    print(f"  {host_names[h]}: {fields_per_host[:, h].mean():.2f}")

## 14. Parallel coordinates — start-year decisions

Each Pareto solution as a polyline of per-field start years, colored by E[ROV]. Highlights how start years shift across the trade-off.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
rov_values = objectives[:, 0]
rov_norm = (rov_values - rov_values.min()) / max(1.0, rov_values.max() - rov_values.min())
cmap = plt.cm.coolwarm

for i in range(n_solutions):
    alloc = allocations[i]
    y = [alloc.start_year[f] if alloc.assignment[f] >= 0 else np.nan
         for f in range(n_fields)]
    ax.plot(range(n_fields), y, color=cmap(rov_norm[i]), alpha=0.4, lw=0.9)

ax.set_xticks(range(n_fields))
ax.set_xticklabels(field_names)
ax.set_ylabel("Start year")
ax.set_title("Start-year decisions across the Pareto front (red = high CVaR, blue = high ROV)")
ax.grid(True, alpha=0.3)

sm = plt.cm.ScalarMappable(cmap=cmap,
                           norm=plt.Normalize(rov_values.min(), rov_values.max()))
sm.set_array([])
plt.colorbar(sm, ax=ax, label="E[ROV] (MNOK)")
plt.tight_layout()
plt.show()

## 15. Cross-seed consistency of field selection

Across all DSBPSO seeds: per-field box plot of selection frequency. Tight boxes mean run-to-run portfolio composition is stable; spread-out boxes mean the choice is sensitive to PSO randomness.